<a href="https://colab.research.google.com/github/vabnamadina/Image-Preprocessing/blob/main/Image_pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

base = "/content/drive/MyDrive/Colab Notebooks/Dataset"

print(os.listdir(base))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Dataset'

In [ ]:
# ============================================================
# SKIN IMAGE PREPROCESSING PIPELINE
# Normal Skin Dataset
# Fix brightness + contrast instead of rejecting
# ============================================================

import os
import cv2
import shutil
import numpy as np
from PIL import Image


# ============================================================
# PATH CONFIGURATION
# ============================================================

RAW_DIR = "/content/drive/MyDrive/Colab Notebooks/Dataset/raw_normal_skin"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Dataset/preprocessed_normal_skin"

IMG_SIZE = (224, 224)


# ============================================================
# IMAGE READING FUNCTION
# ============================================================

def read_image(path):
    ext = os.path.splitext(path)[1].lower()

    if ext in [".webp", ".avif"]:
        img = Image.open(path).convert("RGB")
        img = np.array(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)

    if img is None:
        raise ValueError("Cannot read image")

    return img


# ============================================================
# RESIZE WITH PADDING
# ============================================================

def resize_with_padding(image, size=(224, 224)):
    h, w = image.shape[:2]
    target_w, target_h = size

    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

    x = (target_w - new_w) // 2
    y = (target_h - new_h) // 2

    canvas[y:y + new_h, x:x + new_w] = resized
    return canvas


# ============================================================
# BRIGHTNESS CORRECTION
# ============================================================

def adjust_brightness(image, target_mean=130):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    current_mean = np.mean(gray)

    # Avoid division by zero
    if current_mean < 1:
        current_mean = 1

    # Gamma correction
    gamma = np.log(target_mean / 255.0) / np.log(current_mean / 255.0)
    gamma = np.clip(gamma, 0.5, 2.0)

    lut = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(256)]).astype("uint8")
    corrected = cv2.LUT(image, lut)

    return corrected


# ============================================================
# CONTRAST ENHANCEMENT
# ============================================================

def enhance_contrast(image):
    # Convert to LAB color space
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # CLAHE on L channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)

    merged = cv2.merge((l2, a, b))
    enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

    return enhanced


# ============================================================
# OPTIONAL DENOISING
# ============================================================

def denoise_image(image):
    denoised = cv2.fastNlMeansDenoisingColored(image, None, 5, 5, 7, 21)
    return denoised


# ============================================================
# BLUR CHECK
# Only reject if image is extremely blurry
# ============================================================

def blur_check(image, threshold=5):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur_value = cv2.Laplacian(gray, cv2.CV_64F).var()

    if blur_value < threshold:
        return False, blur_value
    return True, blur_value


# ============================================================
# MAIN PREPROCESSING FUNCTION
# ============================================================

def preprocessing_normal_skin(raw_dir, output_dir):
    if not os.path.exists(raw_dir):
        raise FileNotFoundError(f"Dataset folder not found: {raw_dir}")

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    os.makedirs(output_dir, exist_ok=True)

    images = [
        f for f in os.listdir(raw_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".avif"))
    ]

    print("Total images found:", len(images))

    accepted = 0
    rejected = 0

    for img_name in images:
        img_path = os.path.join(raw_dir, img_name)

        try:
            # 1. Read image
            img = read_image(img_path)

            # 2. Fix brightness
            img = adjust_brightness(img, target_mean=130)

            # 3. Enhance contrast
            img = enhance_contrast(img)

            # 4. Optional denoising
            img = denoise_image(img)

            # 5. Reject only if extremely blurry
            ok_blur, blur_score = blur_check(img, threshold=5)
            if not ok_blur:
                rejected += 1
                continue

            # 6. Resize
            img = resize_with_padding(img, IMG_SIZE)

            # 7. Save output
            save_name = os.path.splitext(img_name)[0] + ".jpg"
            save_path = os.path.join(output_dir, save_name)

            cv2.imwrite(save_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])

            accepted += 1

        except Exception as e:
            print("Error:", img_name, e)
            rejected += 1

    print("\n====================")
    print("Preprocessing Completed")
    print("====================")
    print("Accepted images:", accepted)
    print("Rejected images:", rejected)
    print("Saved at:")
    print(output_dir)


# ============================================================
# RUN PREPROCESSING
# ============================================================

preprocessing_normal_skin(RAW_DIR, OUTPUT_DIR)

FileNotFoundError: Dataset folder not found: /content/drive/MyDrive/Colab Notebooks/Dataset/raw_normal_skin